# 01 — Reconhecimento e Validação das Fontes do Corpus ANEEL

**Objetivo:** validar as 4 fontes de dados do projeto com código Python real, extrair documentos da Wave 1 (~15 docs) e publicar no HuggingFace Hub.

**Por que este notebook existe?**
Via inspeção das requisições de rede do browser (2026-05-28), descobrimos que:
- O diretório `cedoc/` retorna HTTP 403 — não há índice HTML
- O **Power BI** "Gestão do Estoque Regulatório" é o índice real dos atos normativos (API REST)
- **PRODIST/PRORET** estão num GitLab público (`git.aneel.gov.br`)
- **Leis estruturantes** são HTML estático no `planalto.gov.br`

Este notebook valida essas descobertas com código e coleta os primeiros documentos.

**Seções:**
- **A. Power BI API** — consultar o índice de atos normativos (a fonte mais crítica e arriscada)
- **B. Leis estruturantes** — coletar as 4 leis de base do setor elétrico
- **C. RENs via cedoc/** — baixar PDFs das RENs selecionadas na Seção A
- **D. PRODIST via GitLab** — explorar a API do GitLab e baixar o Módulo 1
- **E. Montar DataFrame** — combinar tudo, validar contra o schema, gerar IDs
- **F. Upload para HF Hub** — publicar o corpus Wave 1

---
*Projeto: ANEEL RAG Benchmark | Executa no Google Colab*

## 0. Setup — instalar dependências e configurar ambiente

In [ ]:
# Instala dependências no Colab (no local, use `make install-prod`)
!pip install -q requests curl_cffi beautifulsoup4 lxml PyMuPDF pandas pyarrow huggingface_hub datasets tqdm

import requests
import json
import time
import io
import re
from datetime import datetime, timezone

import pandas as pd
import fitz  # PyMuPDF — extração de texto de PDFs
from bs4 import BeautifulSoup
from tqdm import tqdm

try:
    from curl_cffi import requests as cffi_requests
    import curl_cffi
    print(f"✅ curl_cffi {curl_cffi.__version__} — download do cedoc/ habilitado")
except ImportError as e:
    cffi_requests = None
    print(f"❌ curl_cffi não carregou: {e}")
    print("   Rode: !pip install curl_cffi  →  Runtime → Restart session  →  execute esta célula de novo")

print("✅ Bibliotecas carregadas.")


In [ ]:
# Configuração de secrets — necessário para upload (Seção F)
# No Colab: vá em Secrets (🔑 no painel lateral) e adicione HF_TOKEN.

import os

def _load_secret(name: str) -> str | None:
    """Tenta ler de Colab Secrets primeiro, depois de env var."""
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return os.environ.get(name)

HF_TOKEN = _load_secret("HF_TOKEN")

if HF_TOKEN:
    print("✅ HF_TOKEN carregado.")
else:
    print("⚠️  HF_TOKEN não encontrado. Seção F (upload) não vai funcionar.")


In [ ]:
# --- Constantes do projeto (espelhadas de src/config/settings.py) ---
# Duplicamos aqui para o notebook ser auto-contido no Colab.

ANEEL_CEDOC_URL = "https://www2.aneel.gov.br/cedoc"
ANEEL_POWERBI_URL = (
    "https://wabi-south-central-us-api.analysis.windows.net"
    "/public/reports/querydata?synchronous=true"
)
ANEEL_GITLAB_URL = "https://git.aneel.gov.br"
ANEEL_GITLAB_PROJECT = "publico/centralconteudo"

LEIS_ESTRUTURANTES = {
    "lei-9427-1996": "https://www.planalto.gov.br/ccivil_03/leis/l9427cons.htm",
    "lei-8987-1995": "https://www.planalto.gov.br/ccivil_03/leis/l8987compilada.htm",
    "lei-9074-1995": "https://www.planalto.gov.br/ccivil_03/leis/l9074compilada.htm",
    "lei-13848-2019": "https://www.planalto.gov.br/ccivil_03/_ato2019-2022/2019/lei/l13848.htm",
}

# Headers HTTP — identificação como navegador para evitar bloqueios
HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120.0.0.0 Safari/537.36"
    ),
    "Accept-Language": "pt-BR,pt;q=0.9",
}

# Power BI — identificadores do relatório "Gestão do Estoque Regulatório"
# Descobertos via inspeção das requisições de rede do browser (2026-05-28)
POWERBI_RESOURCE_KEY = "3dcb7cfd-a90c-4d66-8ec3-6934cb4253de"
POWERBI_DATASET_ID = "762a020c-217a-4dae-b9d3-d9b01fd2c14a"
POWERBI_REPORT_ID = "cfda0c11-5d4e-4b61-ad42-7ef82d0be1f6"
POWERBI_MODEL_ID = 5104124

# Repositório HuggingFace
HF_DATASET_REPO = "simoesthiago/aneel-corpus"

# Timestamp da execução
SCRAPED_AT = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")

print(f"✅ Constantes definidas. Timestamp: {SCRAPED_AT}")

---
## A. Power BI API — Índice de Atos Normativos

**Por que começamos pelo Power BI?** É a fonte mais arriscada e mais crítica do projeto. O Power BI é o **único índice estruturado** dos atos normativos da ANEEL — o diretório `cedoc/` retorna 403 sem listagem. Se esta API não funcionar, toda a coleta de atos precisa de um plano B.

**Como a API foi descoberta (inspeção de rede, 2026-05-28):**
O relatório "Gestão do Estoque Regulatório" é um Power BI público. Inspecionando as requisições de rede do browser ao acessar o relatório, identificamos o endpoint REST e o formato do payload. Isso é chamado de *reverse engineering* de uma API não documentada — técnica padrão quando não existe documentação pública.
- Endpoint: POST para `wabi-south-central-us-api.analysis.windows.net/public/reports/querydata`
- Modelo de dados: `DIM Atos Normativos` (Resolução, Situação, Tipo, Ementa, Data)
- ~1.460 atos no total, ~181 RENs vigentes
- Resposta usa formato DSR com dictionary encoding (compressão interna do Power BI)

**O que validamos aqui:**
1. A API aceita nosso payload construído? (sem copiar cookies do browser)
2. Conseguimos decodificar o formato comprimido?
3. Quantos atos existem e quantos são RENs vigentes?
4. Quais RENs selecionar para a Wave 1?

In [ ]:
def consultar_powerbi(propriedades: list[str]) -> dict:
    """
    Consulta a API REST do Power BI público da ANEEL.

    O Power BI expõe relatórios públicos via um endpoint REST que aceita
    "semantic queries" — uma espécie de SQL do Power BI. O payload tem 3
    partes: From (tabelas), Select (colunas), Binding (como agrupar/limitar).

    Args:
        propriedades: lista de nomes de coluna da tabela DIM Atos Normativos.
            Colunas descobertas: "Resolução", "Situação", "Tipo", "Ementa", "Data"

    Returns:
        dict com a resposta JSON completa da API
    """
    # Monta a lista de colunas no formato que o Power BI espera
    select = []
    projections = []
    for i, prop in enumerate(propriedades):
        select.append({
            "Column": {
                "Expression": {"SourceRef": {"Source": "d"}},
                "Property": prop,
            },
            "Name": f"DIM Atos Normativos.{prop}",
        })
        projections.append(i)

    payload = {
        "version": "1.0.0",
        "queries": [
            {
                "Query": {
                    "Commands": [
                        {
                            "SemanticQueryDataShapeCommand": {
                                "Query": {
                                    "Version": 2,
                                    "From": [
                                        {"Name": "d", "Entity": "DIM Atos Normativos", "Type": 0}
                                    ],
                                    "Select": select,
                                },
                                "Binding": {
                                    "Primary": {"Groupings": [{"Projections": projections}]},
                                    "DataReduction": {
                                        "DataVolume": 4,
                                        "Primary": {"Top": {"Count": 30000}},
                                    },
                                    "Version": 1,
                                },
                            }
                        }
                    ]
                },
                "QueryId": "",
                "ApplicationContext": {
                    "DatasetId": POWERBI_DATASET_ID,
                    "Sources": [{"ReportId": POWERBI_REPORT_ID}],
                },
            }
        ],
        "cancelQueries": [],
        "modelId": POWERBI_MODEL_ID,
    }

    headers = {
        "X-PowerBI-ResourceKey": POWERBI_RESOURCE_KEY,
        "Content-Type": "application/json;charset=UTF-8",
    }

    resp = requests.post(ANEEL_POWERBI_URL, headers=headers, json=payload, timeout=30)
    resp.raise_for_status()
    return resp.json()

print("✅ Função consultar_powerbi() definida.")

In [ ]:
def decodificar_dsr(resposta_json: dict, propriedades: list[str]) -> list[dict]:
    """
    Decodifica a resposta DSR (Data Shape Result) do Power BI.

    O Power BI comprime as respostas usando 2 técnicas:
    1. **ValueDicts** — valores repetidos viram índices num dicionário
       (ex.: "Revogada" aparece 1261 vezes → armazenada uma vez no dict D1,
       referenciada como índice 0)
    2. **R bitmask** — colunas que repetem o valor da linha anterior são
       omitidas, e um bitmask indica quais foram omitidas
       (ex.: R=6 = binário 110 → colunas 1 e 2 repetem da linha anterior)

    Args:
        resposta_json: resposta completa da API
        propriedades: lista de nomes de coluna (mesma ordem do Select)

    Returns:
        lista de dicts, um por linha, com as colunas nomeadas
    """
    result = resposta_json["results"][0]["result"]["data"]
    ds = result["dsr"]["DS"][0]
    rows = ds["PH"][0]["DM0"]
    vdicts = ds.get("ValueDicts", {})

    # Schema da resposta — define qual dict (DN) cada coluna usa
    schema = rows[0].get("S", [])
    num_cols = len(schema)

    # Mapeia cada coluna ao seu ValueDict (se houver)
    col_dicts = []
    for s in schema:
        dn = s.get("DN")
        col_dicts.append(vdicts.get(dn) if dn else None)

    # Decodifica linha por linha
    decoded = []
    prev = [None] * num_cols

    for row in rows:
        c = row.get("C", [])
        r_mask = row.get("R", 0)
        current = list(prev)

        # R é um bitmask: bit N setado = coluna N repete da linha anterior
        # Colunas NÃO marcadas em R vêm no array C, na ordem
        c_idx = 0
        for col in range(num_cols):
            bit = 1 << col
            if not (r_mask & bit):  # coluna presente no C
                if c_idx < len(c):
                    current[col] = c[c_idx]
                    c_idx += 1

        prev = list(current)

        # Resolve referências de dicionário (int → string)
        record = {}
        for i, prop in enumerate(propriedades):
            val = current[i]
            d = col_dicts[i]
            if d is not None and isinstance(val, int):
                val = d[val]
            record[prop] = val

        decoded.append(record)

    return decoded

print("✅ Função decodificar_dsr() definida.")

In [ ]:
# --- A.1: Consultar o Power BI com 5 colunas de metadados ---
# Esta é a chamada crítica: se funcionar, temos o índice completo dos atos.
# IMPORTANTE: execute as células 6 e 7 (funções) e a célula 4 (constantes) antes desta.

COLUNAS_POWERBI = ["Resolução", "Situação", "Tipo", "Ementa", "Data"]
atos = []  # garante que A.2 não quebra se esta célula falhar no meio

print("Consultando Power BI API...")
print(f"  Endpoint: {ANEEL_POWERBI_URL}")
print(f"  Colunas:  {COLUNAS_POWERBI}")
print()

try:
    resp_json = consultar_powerbi(COLUNAS_POWERBI)
    atos = decodificar_dsr(resp_json, COLUNAS_POWERBI)
    print(f"✅ Sucesso! {len(atos)} atos retornados.")
except Exception as e:
    print(f"❌ Erro na API: {e}")
    print("   Plano B: usar dataset JvPetas/aneel-legislacao no HF Hub")
    atos = []


In [ ]:
# --- A.2: Analisar a distribuição dos atos ---
# Entender o que tem no Power BI antes de selecionar para a Wave 1

if "atos" not in globals():
    raise RuntimeError(
        "Variável 'atos' não existe. Execute a célula A.1 (acima) antes desta. "
        "No Colab: Runtime → Run before → ou rode as células 4, 6, 7 e 8 em ordem."
    )
if not atos:
    raise RuntimeError(
        "A lista 'atos' está vazia — a célula A.1 falhou ou a API do Power BI não respondeu. "
        "Volte à A.1, leia a mensagem de erro e corrija antes de continuar."
    )

df_atos = pd.DataFrame(atos)

# Converter timestamp (ms desde epoch) para data legível
df_atos["data_formatada"] = pd.to_datetime(df_atos["Data"], unit="ms").dt.strftime("%Y-%m-%d")

# Extrair tipo do ato (REN, RES, DSP, etc.) e número/ano do campo "Resolução"
# Formato: "REN 1000/2021", "RES 798/2002", "DSP 934/2008"
df_atos["sigla"] = df_atos["Resolução"].str.extract(r"^([A-Z]+)\s")
df_atos["numero_raw"] = df_atos["Resolução"].str.extract(r"\s(\d+)/")
df_atos["ano_raw"] = df_atos["Resolução"].str.extract(r"/(\d{4})")

print(f"Total de atos: {len(df_atos)}")
print()
print("=== Distribuição por sigla (tipo de ato) ===")
print(df_atos["sigla"].value_counts().to_string())
print()
print("=== Distribuição por Situação ===")
print(df_atos["Situação"].value_counts().to_string())
print()
print("=== Distribuição por Tipo (classificação) ===")
print(df_atos["Tipo"].value_counts().to_string())


In [ ]:
# --- A.3: Selecionar as ~10 RENs vigentes para a Wave 1 ---
# Critérios: vigentes + tipo "Principal" + temas variados + relevância

# Filtrar RENs vigentes
ren_vigentes = df_atos[
    (df_atos["sigla"] == "REN")
    & (df_atos["Situação"] == "Não consta revogação expressa")
].copy()

print(f"Total de RENs vigentes: {len(ren_vigentes)}")
print()

# Mostrar as 20 mais recentes para seleção manual
ren_vigentes_sorted = ren_vigentes.sort_values("Data", ascending=False)
print("=== 20 RENs vigentes mais recentes (candidatas à Wave 1) ===")
for _, row in ren_vigentes_sorted.head(20).iterrows():
    ementa_curta = str(row["Ementa"])[:90] + "..." if len(str(row["Ementa"])) > 90 else row["Ementa"]
    print(f'  {row["Resolução"]} | {row["Tipo"]} | {row["data_formatada"]}')
    print(f'    {ementa_curta}')
    print()

In [ ]:
# --- A.4: Definir a lista final da Wave 1 ---
# Seleção manual de ~10 RENs que cobrem temas variados do setor elétrico.
# Após rodar A.3, ajuste esta lista conforme os resultados.

RENS_WAVE1 = [
    "REN 1000/2021",   # Regras de Prestação do Serviço de Distribuição
    "REN 1001/2022",   # PRODIST (Módulos 2 e 5, Acesso ao Sistema)
    "REN 1003/2022",   # PRORET (Procedimentos de Regulação Tarifária)
    "REN 1009/2022",   # Contratação de energia (ACR/ACL)
    "REN 1012/2022",   # Procedimentos de Comercialização
    "REN 1059/2023",   # Micro e minigeração distribuída — regras de conexão e faturamento
    "REN 482/2012",    # Micro e minigeração distribuída (marco histórico, substituída pela 1059)
    "REN 414/2010",    # Condições gerais de fornecimento de energia
    "REN 875/2020",    # Estudos de Inventário Hidrelétrico — requisitos e procedimentos
    "REN 956/2021",    # PRODIST — Procedimentos de Distribuição de Energia Elétrica
]

# Filtrar apenas as selecionadas que existem no DataFrame
rens_selecionadas = df_atos[df_atos["Resolução"].isin(RENS_WAVE1)].copy()

print(f"RENs selecionadas para Wave 1: {len(rens_selecionadas)} / {len(RENS_WAVE1)}")
print()

# Verificar quais não foram encontradas (erro de digitação?)
encontradas = set(rens_selecionadas["Resolução"].tolist())
nao_encontradas = set(RENS_WAVE1) - encontradas
if nao_encontradas:
    print(f"⚠️  Não encontradas no Power BI: {nao_encontradas}")
    print()

# Mostrar detalhes das selecionadas
for _, row in rens_selecionadas.iterrows():
    ementa_curta = str(row["Ementa"])[:100] + "..." if len(str(row["Ementa"])) > 100 else row["Ementa"]
    print(f'  {row["Resolução"]} ({row["data_formatada"]}) — {row["Situação"]}')
    print(f'    {ementa_curta}')
    print()

---
## B. Leis Estruturantes (planalto.gov.br)

Fonte mais simples do projeto: 4 leis em HTML estático no site do Planalto.

**Riscos:**
- Encoding pode ser ISO-8859-1 (sites governamentais antigos)
- HTML pode ter muita navegação/footer a remover
- Texto pode incluir notas de rodapé e links que poluem a extração

In [ ]:
def extrair_texto_lei(url: str) -> dict:
    """
    Baixa uma lei do planalto.gov.br e extrai o texto limpo.

    O site do Planalto usa encoding ISO-8859-1 em algumas páginas.
    O corpo da lei fica dentro de tags <p> no conteúdo principal.
    Navegação, headers e footers precisam ser removidos.

    Returns:
        dict com: texto, encoding_detectado, num_chars, sucesso
    """
    resp = requests.get(url, headers=HEADERS, timeout=30)
    resp.raise_for_status()

    # Detectar encoding — o Planalto às vezes usa ISO-8859-1
    # O requests tenta adivinhar, mas nem sempre acerta
    encoding_detectado = resp.encoding
    if resp.encoding and "iso" in resp.encoding.lower():
        resp.encoding = resp.apparent_encoding or "utf-8"

    soup = BeautifulSoup(resp.text, "lxml")

    # Remover scripts, styles e navegação
    for tag in soup.find_all(["script", "style", "nav", "header", "footer"]):
        tag.decompose()

    # O corpo da lei no Planalto geralmente está em <p> dentro do conteúdo
    # Estratégia: pegar todo o texto visível e limpar
    paragrafos = soup.find_all("p")
    texto_partes = []
    for p in paragrafos:
        texto = p.get_text(strip=True)
        if texto and len(texto) > 10:  # ignora parágrafos muito curtos (navegação)
            texto_partes.append(texto)

    texto_final = "\n\n".join(texto_partes)

    return {
        "texto": texto_final,
        "encoding_detectado": encoding_detectado,
        "num_chars": len(texto_final),
        "sucesso": len(texto_final) > 1000,  # lei tem que ter conteúdo substancial
    }


# --- B.1: Coletar as 4 leis ---
# Mapeamento: id → metadados completos para o schema

LEIS_METADATA = {
    "lei-9427-1996": {
        "titulo": "Lei 9.427/1996 — Criação da ANEEL",
        "numero": "9427",
        "ano": 1996,
    },
    "lei-8987-1995": {
        "titulo": "Lei 8.987/1995 — Concessões de Serviços Públicos",
        "numero": "8987",
        "ano": 1995,
    },
    "lei-9074-1995": {
        "titulo": "Lei 9.074/1995 — Outorgas e Prorrogações de Concessões",
        "numero": "9074",
        "ano": 1995,
    },
    "lei-13848-2019": {
        "titulo": "Lei 13.848/2019 — Lei das Agências Reguladoras",
        "numero": "13848",
        "ano": 2019,
    },
}

documentos_leis = []

for lei_id, url in LEIS_ESTRUTURANTES.items():
    meta = LEIS_METADATA[lei_id]
    print(f"Coletando {lei_id}...")

    try:
        resultado = extrair_texto_lei(url)
        print(f"  ✅ {resultado['num_chars']} chars | encoding: {resultado['encoding_detectado']}")

        documentos_leis.append({
            "id": lei_id,
            "tipo": "lei",
            "subtipo": "lei_federal",
            "numero": meta["numero"],
            "ano": meta["ano"],
            "titulo": meta["titulo"],
            "assunto": None,
            "situacao": None,  # leis não têm "situação" como atos
            "data_publicacao": None,
            "fonte": "planalto",
            "url_original": url,
            "url_consolidado": None,
            "formato_original": "html",
            "texto_bruto": resultado["texto"],
            "num_paginas": None,
            "metodo_extracao": "html_parser",
            "qualidade_extracao": 1.0 if resultado["sucesso"] else 0.5,
            "hf_path": None,
            "scraped_at": SCRAPED_AT,
        })

    except Exception as e:
        print(f"  ❌ Erro: {e}")

    time.sleep(1)  # respeito ao servidor

print(f"\n✅ {len(documentos_leis)} leis coletadas com sucesso.")

In [ ]:
# --- B.2: Verificar qualidade da extração ---
# Uma olhada rápida no texto para ver se a extração faz sentido

for doc in documentos_leis:
    print(f'=== {doc["id"]} ({doc["num_paginas"]} págs, {len(doc["texto_bruto"])} chars) ===')
    # Mostra os primeiros 300 chars do texto
    preview = doc["texto_bruto"][:300].replace("\n", " ")
    print(f"  {preview}...")
    print()

---
## C. RENs via cedoc/ — Download direto

Padrão de URL (**sem** zero-padding no número):
- Original: `https://www2.aneel.gov.br/cedoc/ren{ano}{numero}.pdf`
- Consolidada: `https://www2.aneel.gov.br/cedoc/bren{ano}{numero}.pdf`

Ex.: REN 414/2010 → `ren2010414.pdf` (não `ren20100414.pdf`).

O `cedoc/` exige TLS de browser real → usamos `curl_cffi` (`impersonate="chrome120"`).

**Colab:** o IP do Google Cloud costuma receber **HTTP 403** do Cloudflare mesmo com URL correta.
A célula **C.0** abaixo mostra o diagnóstico. Se der 403:
1. Rode a ingestão em **GitHub Actions** (`ingest_corpus.yml`) ou na sua máquina local; ou
2. Pule a Seção C e carregue o Parquet já publicado no HF Hub na Seção E.


In [ ]:
# --- C.0: Preflight — testar acesso ao cedoc/ (rode antes do C.1) ---
if cffi_requests is None:
    raise RuntimeError(
        "curl_cffi não está disponível. Instale com !pip install curl_cffi, "
        "reinicie o runtime (Runtime → Restart session) e execute o Setup de novo."
    )

_url_teste = f"{ANEEL_CEDOC_URL}/ren20211000.pdf"
print(f"URL de teste: {_url_teste}")
print()

_impersonates = ["chrome120", "chrome124", "chrome", "safari17_0"]
_ok = False
for _imp in _impersonates:
    try:
        _r = cffi_requests.get(_url_teste, impersonate=_imp, timeout=60)
        _eh_pdf = _r.status_code == 200 and len(_r.content) > 1000 and _r.content[:4] == b"%PDF"
        print(f"  impersonate={_imp!r:14} → HTTP {_r.status_code} | {len(_r.content):>8} bytes | PDF={_eh_pdf}")
        if _eh_pdf:
            _ok = True
            CEDOC_IMPERSONATE = _imp
            break
    except Exception as _e:
        print(f"  impersonate={_imp!r:14} → erro: {_e}")

print()
if _ok:
    print(f"✅ cedoc/ acessível neste ambiente (usando {_imp!r}). Pode rodar a célula C.1.")
else:
    print("❌ Nenhum impersonate funcionou neste ambiente.")
    print("   Causa mais comum no Colab: Cloudflare bloqueia IP de datacenter (403).")
    print("   O código e as URLs estão corretos — o bloqueio é de rede, não de formato.")
    print()
    print("   Alternativas:")
    print("   • GitHub Actions → workflow 'Ingest ANEEL corpus' (wave=1)")
    print("   • Local: python -m src.ingestion.run_wave --wave 1")
    print("   • Carregar corpus já publicado: datasets.load_dataset('simoesthiago/aneel-corpus')")
    CEDOC_IMPERSONATE = "chrome120"  # fallback para C.1 tentar mesmo assim


In [ ]:
def montar_url_ato(sigla: str, ano: int, numero: int) -> str:
    """
    Constrói a URL do PDF no cedoc/ a partir da sigla, ano e número.

    Padrão: https://www2.aneel.gov.br/cedoc/{sigla_lower}{ano}{numero}.pdf
    O cedoc/ NÃO usa zero-padding no número.

    Exemplos:
      REN 1000/2021 → ren20211000.pdf
      REN 414/2010  → ren2010414.pdf
    """
    sigla_lower = sigla.lower()
    return f"{ANEEL_CEDOC_URL}/{sigla_lower}{ano}{numero}.pdf"


def montar_url_consolidada(sigla: str, ano: int, numero: int) -> str:
    """URL da versão consolidada (prefixo 'b'). Também sem zero-padding."""
    sigla_lower = sigla.lower()
    return f"{ANEEL_CEDOC_URL}/b{sigla_lower}{ano}{numero}.pdf"


def extrair_texto_pdf(conteudo: bytes) -> dict:
    """
    Extrai texto de um PDF usando PyMuPDF (fitz).

    Processa o PDF inteiramente em memória — NUNCA salva em disco.
    Calcula qualidade_extracao como a fração de páginas com texto substancial.

    Args:
        conteudo: bytes do PDF

    Returns:
        dict com: texto, num_paginas, qualidade_extracao, chars_por_pagina
    """
    pdf = fitz.open(stream=io.BytesIO(conteudo), filetype="pdf")
    num_paginas = len(pdf)

    textos_paginas = []
    paginas_com_texto = 0

    for pagina in pdf:
        texto = pagina.get_text()
        textos_paginas.append(texto)
        # Página com mais de 100 chars = provavelmente digital (não escaneada)
        if len(texto.strip()) > 100:
            paginas_com_texto += 1

    pdf.close()

    texto_completo = "\n\n".join(textos_paginas)
    qualidade = paginas_com_texto / num_paginas if num_paginas > 0 else 0.0

    return {
        "texto": texto_completo,
        "num_paginas": num_paginas,
        "qualidade_extracao": round(qualidade, 2),
        "chars_por_pagina": len(texto_completo) / num_paginas if num_paginas > 0 else 0,
    }


print("✅ Funções de URL e extração de PDF definidas.")


In [ ]:
# --- C.1: Baixar e extrair texto das RENs selecionadas ---

_IMP = globals().get("CEDOC_IMPERSONATE", "chrome120")


def baixar_pdf_cedoc(url: str, timeout: int = 60) -> bytes | None:
    """
    Baixa PDF do cedoc/ com curl_cffi. Igual ao scraper de produção (scraper_atos.py).
    """
    if cffi_requests is None:
        print("    curl_cffi indisponível")
        return None
    try:
        resp = cffi_requests.get(url, impersonate=_IMP, timeout=timeout)
        if resp.status_code == 200 and len(resp.content) > 1000 and resp.content[:4] == b"%PDF":
            return resp.content
        # Diagnóstico — essencial para entender 403 no Colab vs 404 por URL errada
        inicio = resp.content[:60]
        try:
            inicio_txt = inicio.decode("utf-8", errors="replace").replace("\n", " ")
        except Exception:
            inicio_txt = repr(inicio)
        print(
            f"    HTTP {resp.status_code} | {len(resp.content)} bytes | "
            f"início: {inicio_txt[:80]!r}"
        )
    except Exception as e:
        print(f"    erro de rede: {e}")
    return None


documentos_rens = []

for _, row in rens_selecionadas.iterrows():
    resolucao = row["Resolução"]

    match = re.match(r"(\w+)\s+(\d+)/(\d{4})", resolucao)
    if not match:
        print(f"  ⚠️  Formato inesperado: {resolucao}")
        continue

    sigla, numero_str, ano_str = match.groups()
    numero = int(numero_str)
    ano = int(ano_str)

    url_original = montar_url_ato(sigla, ano, numero)
    url_consolidada = montar_url_consolidada(sigla, ano, numero)

    print(f"Baixando {resolucao}...")
    print(f"  URL original: {url_original}")

    pdf_bytes = None
    url_usada = None
    for url_tentativa, label in [(url_consolidada, "consolidada"), (url_original, "original")]:
        pdf_bytes_candidato = baixar_pdf_cedoc(url_tentativa)
        if pdf_bytes_candidato:
            pdf_bytes = pdf_bytes_candidato
            url_usada = url_tentativa
            print(f"  ✅ {label}: {len(pdf_bytes) / 1024:.0f} KB")
            break
        if label == "consolidada":
            print(f"  ⚠️  {label}: falhou (tentando original...)")
        else:
            print(f"  ⚠️  {label}: falhou")

    if pdf_bytes is None:
        print(f"  ❌ Nenhuma versão encontrada para {resolucao}")
        continue

    try:
        resultado = extrair_texto_pdf(pdf_bytes)
        print(
            f"  📄 {resultado['num_paginas']} págs | "
            f"{len(resultado['texto'])} chars | "
            f"qualidade: {resultado['qualidade_extracao']}"
        )

        data_pub = None
        if row["Data"] and not pd.isna(row["Data"]):
            data_pub = pd.to_datetime(row["Data"], unit="ms").strftime("%Y-%m-%d")

        doc_id = f"{sigla.lower()}-{ano}-{numero}"

        documentos_rens.append({
            "id": doc_id,
            "tipo": "ato_normativo",
            "subtipo": sigla.lower(),
            "numero": numero_str,
            "ano": ano,
            "titulo": row["Ementa"] if row["Ementa"] else resolucao,
            "assunto": None,
            "situacao": "vigente",
            "data_publicacao": data_pub,
            "fonte": "cedoc",
            "url_original": url_original,
            "url_consolidado": url_consolidada if url_usada == url_consolidada else None,
            "formato_original": "pdf",
            "texto_bruto": resultado["texto"],
            "num_paginas": resultado["num_paginas"],
            "metodo_extracao": "pymupdf",
            "qualidade_extracao": resultado["qualidade_extracao"],
            "hf_path": None,
            "scraped_at": SCRAPED_AT,
        })

    except Exception as e:
        print(f"  ❌ Erro na extração: {e}")

    time.sleep(2)

print(f"\n✅ {len(documentos_rens)} RENs processadas com sucesso.")
if len(documentos_rens) == 0:
    print("⚠️  Se todas falharam com HTTP 403, o Colab está bloqueado — veja a célula C.0.")


In [ ]:
# --- C.alt: Fallback quando o Colab recebe HTTP 403 no cedoc/ ---
# O Cloudflare bloqueia IPs de datacenter (Google Cloud). URLs e curl_cffi estão corretos.
# Use esta célula para puxar as RENs já ingeridas em outro ambiente (Actions ou local).

if documentos_rens:
    print(f"✅ Já há {len(documentos_rens)} RENs em documentos_rens — fallback não necessário.")
else:
    print("documentos_rens vazio — tentando HuggingFace Hub...")
    _repo = HF_DATASET_REPO
    _ids_wave1 = []
    for _r in RENS_WAVE1:
        _m = re.match(r"REN\s+(\d+)/(\d{4})", _r)
        if _m:
            _ids_wave1.append(f"ren-{_m.group(2)}-{_m.group(1)}")

    try:
        from datasets import load_dataset

        ds = load_dataset(_repo, split="train")
        df_rens_hf = ds.to_pandas()
        df_rens_hf = df_rens_hf[
            (df_rens_hf["tipo"] == "ato_normativo") & (df_rens_hf["id"].isin(_ids_wave1))
        ]
        documentos_rens = df_rens_hf.to_dict(orient="records")
        print(f"✅ {len(documentos_rens)} / {len(_ids_wave1)} RENs carregadas de {_repo}")
        _faltando = set(_ids_wave1) - {d["id"] for d in documentos_rens}
        if _faltando:
            print(f"⚠️  IDs ainda não no Hub: {_faltando}")
    except Exception as e:
        print(f"❌ Não foi possível carregar do HF Hub: {e}")
        print()
        print("Rode a ingestão FORA do Colab, depois execute esta célula de novo:")
        print("  • GitHub Actions → 'Ingest ANEEL corpus' → wave=1  (HF_TOKEN nos secrets)")
        print("  • OU local: python -m src.ingestion.run_wave --wave 1")
        print()
        print("Você pode montar o DataFrame na Seção E só com leis + PRODIST até lá.")


---
## D. PRODIST via GitLab (git.aneel.gov.br)

O PRODIST (Procedimentos de Distribuição) e outros procedimentos regulatórios ficam num GitLab público da ANEEL. Para a Wave 1, vamos coletar o **Módulo 1 do PRODIST** (Introdução).

**API REST do GitLab:**
- Listagem: `GET /api/v4/projects/{id}/repository/tree?path=...`
- Download: `GET /api/v4/projects/{id}/repository/files/{path}/raw`

**Limitação conhecida:** o GitLab não exige autenticação para projetos públicos, mas o IP do Google Cloud (Colab) costuma receber **timeout de conexão** — o servidor parece bloquear IPs de datacenter estrangeiros, assim como o cedoc/. Se D.1 falhar com timeout, a coleta do PRODIST deve ser feita via GitHub Actions (`ingest_corpus.yml`).

In [ ]:
# --- D.1: Explorar a estrutura do repositório GitLab ---
# Primeiro, descobrir o ID numérico do projeto e a estrutura de pastas.

import urllib.parse

# O GitLab identifica projetos pelo ID numérico ou pelo path URL-encoded
GITLAB_PROJECT_PATH = urllib.parse.quote(ANEEL_GITLAB_PROJECT, safe="")

# Listar conteúdo na raiz do repositório
print("Explorando a raiz do repositório ANEEL no GitLab...")
print(f"  URL base: {ANEEL_GITLAB_URL}")
print(f"  Projeto: {ANEEL_GITLAB_PROJECT}")
print()

try:
    url_tree = f"{ANEEL_GITLAB_URL}/api/v4/projects/{GITLAB_PROJECT_PATH}/repository/tree"
    resp = requests.get(url_tree, params={"per_page": 50}, timeout=30)
    resp.raise_for_status()
    itens_raiz = resp.json()

    print(f"✅ API acessível! {len(itens_raiz)} itens na raiz:")
    for item in itens_raiz:
        tipo = "📁" if item["type"] == "tree" else "📄"
        print(f"  {tipo} {item['name']} ({item['type']})")

except Exception as e:
    print(f"❌ Erro ao acessar GitLab: {e}")
    print("   Verificar se o projeto existe e é público.")
    itens_raiz = []

In [ ]:
# --- D.2: Navegar até o PRODIST e encontrar o Módulo 1 ---
# A estrutura pode ser algo como: PRODIST/ → Módulo 1/ → arquivo.pdf

def listar_gitlab(path: str = "") -> list[dict]:
    """Lista arquivos/pastas em um caminho do repositório GitLab."""
    url = f"{ANEEL_GITLAB_URL}/api/v4/projects/{GITLAB_PROJECT_PATH}/repository/tree"
    params = {"path": path, "per_page": 100}
    resp = requests.get(url, params=params, timeout=30)
    resp.raise_for_status()
    return resp.json()


# Procurar pasta PRODIST (nome pode variar)
print("Procurando pasta PRODIST...")
prodist_path = None

for item in itens_raiz:
    if "prodist" in item["name"].lower() and item["type"] == "tree":
        prodist_path = item["path"]
        print(f"  ✅ Encontrada: {prodist_path}")
        break

if prodist_path is None:
    # Tentar um nível mais fundo
    print("  Não encontrada na raiz. Procurando em subpastas...")
    for item in itens_raiz:
        if item["type"] == "tree":
            try:
                sub_itens = listar_gitlab(item["path"])
                for sub in sub_itens:
                    if "prodist" in sub["name"].lower() and sub["type"] == "tree":
                        prodist_path = sub["path"]
                        print(f"  ✅ Encontrada: {prodist_path}")
                        break
            except Exception:
                pass
        if prodist_path:
            break

if prodist_path:
    print(f"\nConteúdo de {prodist_path}/:")
    prodist_itens = listar_gitlab(prodist_path)
    for item in prodist_itens:
        tipo = "📁" if item["type"] == "tree" else "📄"
        print(f"  {tipo} {item['name']}")
else:
    print("  ❌ Pasta PRODIST não encontrada. Estrutura do repo pode ser diferente.")

In [ ]:
# --- D.3: Baixar o Módulo 1 do PRODIST ---
# Adapte o caminho abaixo conforme o que D.2 revelar.

# IMPORTANTE: Preencha estas variáveis após executar D.2
# O caminho exato depende da estrutura real do repositório.
# Exemplo esperado: "PRODIST/Módulo 1/PRODIST - Módulo 1 - Revisão X.pdf"
PRODIST_MODULO1_PATH = None  # ← Preencher após D.2

documentos_prodist = []

if prodist_path:
    # Tentar encontrar o Módulo 1 automaticamente
    print("Procurando Módulo 1 do PRODIST...")

    modulo1_path = None
    for item in prodist_itens:
        nome_lower = item["name"].lower()
        if ("módulo 1" in nome_lower or "modulo 1" in nome_lower or "modulo1" in nome_lower):
            if item["type"] == "tree":
                # É uma pasta — listar conteúdo para achar o PDF
                sub_itens = listar_gitlab(item["path"])
                for sub in sub_itens:
                    if sub["name"].lower().endswith(".pdf"):
                        modulo1_path = sub["path"]
                        break
            elif item["name"].lower().endswith(".pdf"):
                modulo1_path = item["path"]
            break

    if modulo1_path:
        print(f"  ✅ Encontrado: {modulo1_path}")

        # Download via API do GitLab (raw file)
        file_path_encoded = urllib.parse.quote(modulo1_path, safe="")
        url_download = (
            f"{ANEEL_GITLAB_URL}/api/v4/projects/{GITLAB_PROJECT_PATH}"
            f"/repository/files/{file_path_encoded}/raw"
        )
        print(f"  Baixando...")
        resp = requests.get(url_download, params={"ref": "master"}, timeout=120)

        if resp.status_code == 200:
            pdf_bytes = resp.content
            print(f"  ✅ {len(pdf_bytes) / 1024:.0f} KB baixados")

            resultado = extrair_texto_pdf(pdf_bytes)
            print(f"  📄 {resultado['num_paginas']} págs | "
                  f"{len(resultado['texto'])} chars | "
                  f"qualidade: {resultado['qualidade_extracao']}")

            documentos_prodist.append({
                "id": "prodist-modulo-01",
                "tipo": "procedimento",
                "subtipo": "prodist",
                "numero": "Módulo 1",
                "ano": None,  # PRODIST é atualizado continuamente
                "titulo": "PRODIST — Módulo 1 — Introdução",
                "assunto": "Procedimentos de Distribuição",
                "situacao": None,
                "data_publicacao": None,
                "fonte": "gitlab",
                "url_original": f"{ANEEL_GITLAB_URL}/{ANEEL_GITLAB_PROJECT}/-/blob/master/{modulo1_path}",
                "url_consolidado": None,
                "formato_original": "pdf",
                "texto_bruto": resultado["texto"],
                "num_paginas": resultado["num_paginas"],
                "metodo_extracao": "pymupdf",
                "qualidade_extracao": resultado["qualidade_extracao"],
                "hf_path": None,
                "scraped_at": SCRAPED_AT,
            })
        else:
            print(f"  ❌ Erro HTTP {resp.status_code}. Tentar ref='main' em vez de 'master'?")
    else:
        print("  ⚠️  Módulo 1 não encontrado automaticamente.")
        print("  Verifique a saída de D.2 e preencha PRODIST_MODULO1_PATH manualmente.")
else:
    print("⚠️  Seção D.2 não encontrou o PRODIST. Pule esta célula ou investigue.")

print(f"\n✅ {len(documentos_prodist)} documento(s) do PRODIST coletado(s).")

---
## E. Montar DataFrame — Consolidar e Validar

Aqui combinamos todos os documentos coletados nas Seções B, C e D num único DataFrame, validamos contra o schema definido em `docs/schema.md`, e preparamos para upload.

In [ ]:
# --- E.1: Combinar todos os documentos ---

todos_documentos = documentos_leis + documentos_rens + documentos_prodist

print(f"Total de documentos coletados: {len(todos_documentos)}")
print(f"  Leis:      {len(documentos_leis)}")
print(f"  RENs:      {len(documentos_rens)}")
print(f"  PRODIST:   {len(documentos_prodist)}")
print()

# Criar DataFrame
df = pd.DataFrame(todos_documentos)

# Verificar schema — todas as colunas obrigatórias presentes?
SCHEMA_COLUNAS = [
    "id", "tipo", "subtipo", "numero", "ano", "titulo", "assunto",
    "situacao", "data_publicacao", "fonte", "url_original", "url_consolidado",
    "formato_original", "texto_bruto", "num_paginas", "metodo_extracao",
    "qualidade_extracao", "hf_path", "scraped_at",
]

colunas_faltando = set(SCHEMA_COLUNAS) - set(df.columns)
colunas_extras = set(df.columns) - set(SCHEMA_COLUNAS)

if colunas_faltando:
    print(f"❌ Colunas faltando: {colunas_faltando}")
elif colunas_extras:
    print(f"⚠️  Colunas extras (serão removidas): {colunas_extras}")
    df = df[SCHEMA_COLUNAS]
else:
    print("✅ Schema validado — todas as colunas presentes!")

# Reordenar colunas conforme o schema
df = df[SCHEMA_COLUNAS]

print()
print("=== Resumo do DataFrame ===")
print(df[["id", "tipo", "fonte", "formato_original", "qualidade_extracao"]].to_string())
print()
print(f"Tamanho total do texto: {df['texto_bruto'].str.len().sum():,.0f} chars")
print(f"Qualidade média: {df['qualidade_extracao'].mean():.2f}")

In [ ]:
# --- E.2: Validações de integridade ---

print("=== Validações ===")
erros = []

# 1. Nenhum texto_bruto vazio
vazios = df[df["texto_bruto"].str.len() < 100]
if len(vazios) > 0:
    erros.append(f"❌ {len(vazios)} documentos com texto_bruto < 100 chars: {vazios['id'].tolist()}")
else:
    print("✅ Nenhum texto_bruto vazio")

# 2. IDs únicos
duplicados = df[df["id"].duplicated()]
if len(duplicados) > 0:
    erros.append(f"❌ IDs duplicados: {duplicados['id'].tolist()}")
else:
    print("✅ Todos os IDs são únicos")

# 3. Qualidade de extração aceitável (> 0.5)
baixa_qualidade = df[df["qualidade_extracao"] < 0.5]
if len(baixa_qualidade) > 0:
    erros.append(f"⚠️  {len(baixa_qualidade)} docs com qualidade < 0.5: {baixa_qualidade['id'].tolist()}")
else:
    print("✅ Todos os documentos com qualidade >= 0.5")

# 4. Campos obrigatórios não-nulos
for col in ["id", "tipo", "titulo", "fonte", "url_original", "formato_original", "texto_bruto", "metodo_extracao", "scraped_at"]:
    nulos = df[df[col].isna()]
    if len(nulos) > 0:
        erros.append(f"❌ Coluna obrigatória '{col}' tem {len(nulos)} nulos")

if not erros:
    print("✅ Todas as validações passaram!")
else:
    print()
    for e in erros:
        print(e)

---
## F. Upload para HuggingFace Hub

Publicar o corpus Wave 1 como um dataset Parquet no HuggingFace Hub.

**Pré-requisito:** o repositório `simoesthiago/aneel-corpus` deve existir no HF Hub e o `HF_TOKEN` deve estar configurado (ver Seção 0).

**Particionamento:** por `tipo` (ato_normativo, procedimento, lei) para queries eficientes.

In [ ]:
# --- F.1: Salvar como Parquet e fazer upload ---

from huggingface_hub import HfApi

if HF_TOKEN is None:
    print("❌ HF_TOKEN não configurado. Pule esta célula.")
    print("   Configure em: Colab → Secrets → HF_TOKEN = hf_...")
else:
    api = HfApi(token=HF_TOKEN)

    # Salvar Parquet particionado por tipo em diretório temporário
    import tempfile
    import os

    with tempfile.TemporaryDirectory() as tmpdir:
        # Salvar um Parquet por tipo (particionamento manual)
        for tipo in df["tipo"].unique():
            df_tipo = df[df["tipo"] == tipo]
            path = os.path.join(tmpdir, f"data/documents/tipo={tipo}/part-0.parquet")
            os.makedirs(os.path.dirname(path), exist_ok=True)
            df_tipo.drop(columns=["tipo"]).to_parquet(path, index=False, engine="pyarrow")
            print(f"  📦 {path}: {len(df_tipo)} documentos")

        # Upload para o HF Hub
        print(f"\nFazendo upload para {HF_DATASET_REPO}...")
        try:
            api.upload_folder(
                folder_path=tmpdir,
                repo_id=HF_DATASET_REPO,
                repo_type="dataset",
                commit_message=f"Wave 1: {len(df)} documentos ({SCRAPED_AT})",
            )
            print(f"✅ Upload concluído! Verifique: https://huggingface.co/datasets/{HF_DATASET_REPO}")
        except Exception as e:
            print(f"❌ Erro no upload: {e}")
            print("   Verifique se o repositório existe e se o token tem permissão de escrita.")

In [ ]:
# --- F.2: Verificar leitura de volta (sanity check) ---
# Se o upload funcionou, vamos ler o dataset de volta para confirmar.

from datasets import load_dataset

if HF_TOKEN:
    try:
        ds = load_dataset(HF_DATASET_REPO, token=HF_TOKEN)
        print(f"✅ Dataset lido com sucesso do HF Hub!")
        print(f"   Splits: {list(ds.keys())}")
        for split_name, split_data in ds.items():
            print(f"   {split_name}: {len(split_data)} documentos, {split_data.column_names}")
    except Exception as e:
        print(f"❌ Erro ao ler de volta: {e}")
        print("   O dataset pode levar alguns segundos para ficar disponível.")
else:
    print("⚠️  Pular verificação (HF_TOKEN não configurado).")

---
## Resumo e Próximos Passos

### O que este notebook validou:
1. **Power BI API funciona** — sem autenticação, retorna ~1.460 atos com 5 colunas de metadados
2. **Leis do Planalto** — HTML parseável, encoding ISO-8859-1 tratável, 4 leis coletadas
3. **PDFs do cedoc/** — padrão de URL sem zero-padding confirmado, PyMuPDF extrai texto
4. **GitLab ANEEL** — API REST pública, mas IPs do Colab recebem timeout (usar Actions)

### Limitações do Colab para este projeto:
- **cedoc/** bloqueia IPs do Google Cloud → usar GitHub Actions com Worker CF + Smart Placement
- **GitLab ANEEL** tem timeout do Colab → mesmo caminho via Actions

### Pipeline de produção (substituiu o notebook para execução recorrente):
- `src/ingestion/` — scrapers implementados e testados (39 testes)
- `.github/workflows/ingest_corpus.yml` — cron mensal + dispatch manual
- Cloudflare Worker `aneel-proxy` + **Smart Placement** → cedoc/ via edge brasileira
- Dataset publicado em `huggingface.co/datasets/simoesthiago/aneel-corpus`

### Estado atual do corpus:
- **Wave 1 (concluída):** 4 leis + 10 RENs = ~14 documentos no HF Hub
- **Wave 2 (próxima):** todas as ~181 RENs vigentes
- **Wave 3 (futura):** todos os ~1.460 atos + PRODIST/PRORET completos + manuais